# Demo 5: Building Agents

This demo covers building a single agent with UC function tools. We create one agent with a system prompt listing available tools, then test it with 3 different questions to validate the agent picks the right tool each time.

In [0]:
%sql
-- SETUP: Create catalog, schema, sample data, and UC functions
-- for building and testing a single agent.

CREATE CATALOG IF NOT EXISTS module5a_demo5;
CREATE SCHEMA IF NOT EXISTS module5a_demo5.agent_tools;

-- Sample support tickets table
CREATE OR REPLACE TABLE module5a_demo5.agent_tools.support_tickets (
  ticket_id    STRING NOT NULL,
  customer_id  STRING NOT NULL,
  category     STRING,
  priority     STRING,
  subject      STRING,
  description  STRING,
  status       STRING,
  created_at   TIMESTAMP
);

INSERT INTO module5a_demo5.agent_tools.support_tickets VALUES
('TKT-001', 'CUST-001', 'billing',   'high',   'Overcharged on order ORD-001', 'I was charged $249.99 but the product was on sale for $199.99', 'open', '2026-09-25 08:00:00'),
('TKT-002', 'CUST-002', 'shipping', 'medium', 'Order ORD-003 not delivered',   'My order from Sept 20 has not arrived yet',                     'open', '2026-09-25 09:30:00'),
('TKT-003', 'CUST-003', 'technical','high',   'Cannot access my account',     'I keep getting a 403 error when trying to log in',             'open', '2026-09-25 10:15:00'),
('TKT-004', 'CUST-001', 'general',   'low',    'Product recommendation',      'Can you recommend accessories for my recent purchase?',        'open', '2026-09-25 11:00:00'),
('TKT-005', 'CUST-002', 'billing',   'medium', 'Discount not applied',        'My premium discount was not applied to order ORD-008',         'open', '2026-09-25 11:45:00');

-- UC functions as agent tools
CREATE OR REPLACE FUNCTION module5a_demo5.agent_tools.get_ticket(ticket_id STRING)
RETURNS STRING
COMMENT 'Returns the subject and description of a support ticket'
RETURN (SELECT max(concat_ws(' | ', subject, description))
       FROM module5a_demo5.agent_tools.support_tickets
       WHERE ticket_id = get_ticket.ticket_id);

CREATE OR REPLACE FUNCTION module5a_demo5.agent_tools.count_tickets_by_category(cat STRING)
RETURNS INT
COMMENT 'Returns the number of tickets in a given category'
RETURN SELECT count(*) FROM module5a_demo5.agent_tools.support_tickets
       WHERE category = count_tickets_by_category.cat;

CREATE OR REPLACE FUNCTION module5a_demo5.agent_tools.get_priority(ticket_id STRING)
RETURNS STRING
COMMENT 'Returns the priority of a support ticket'
RETURN (SELECT max(priority) FROM module5a_demo5.agent_tools.support_tickets
       WHERE ticket_id = get_priority.ticket_id);

SELECT * FROM module5a_demo5.agent_tools.support_tickets ORDER BY ticket_id;

ticket_id,customer_id,category,priority,subject,description,status,created_at
TKT-001,CUST-001,billing,high,Overcharged on order ORD-001,I was charged .99 but the product was on sale for .99,open,2026-09-25T08:00:00.000Z
TKT-002,CUST-002,shipping,medium,Order ORD-003 not delivered,My order from Sept 20 has not arrived yet,open,2026-09-25T09:30:00.000Z
TKT-003,CUST-003,technical,high,Cannot access my account,I keep getting a 403 error when trying to log in,open,2026-09-25T10:15:00.000Z
TKT-004,CUST-001,general,low,Product recommendation,Can you recommend accessories for my recent purchase?,open,2026-09-25T11:00:00.000Z
TKT-005,CUST-002,billing,medium,Discount not applied,My premium discount was not applied to order ORD-008,open,2026-09-25T11:45:00.000Z


## Agent Anatomy: The Reason-Act-Observe Loop

### Concepts
An AI agent is not just a single LLM call. It follows a loop:

1. **Reason**: The LLM analyzes the user's request and decides what to do next.
   - "The user wants to know about TKT-001. I should call the get_ticket tool."

2. **Act**: The agent executes a tool (UC function, API call, SQL query).
   - Calls `get_ticket('TKT-001')` -> "Overcharged on order ORD-001 | I was charged..."

3. **Observe**: The agent sees the tool's output and incorporates it.
   - "The ticket is about a billing overcharge. The priority is high."

4. **Respond**: The agent synthesizes a natural language response.
   - "Ticket TKT-001 is a high-priority billing issue about an overcharge on order ORD-001."

**Key components of an agent**:
* **System prompt**: Defines the agent's role, capabilities, and constraints
* **Tools**: Functions the agent can call (UC functions, APIs, code)
* **Reasoning loop**: The reason-act-observe cycle that continues until the agent has enough information to respond

> A single LLM call is a question-answer pair. An agent is a **sequence of decisions** that may involve multiple tool calls before responding.

## 5.1 : Single-Agent vs. Multi-Agent

### When to use a single agent
A single agent is the right choice when:
* **Few tools** (< 8): The LLM can manage all tools effectively
* **One domain**: All questions fall within one area of expertise
* **Short instructions**: The system prompt fits comfortably in context
* **Simple flow**: The reason-act-observe loop doesn't need delegation

### When to split into multi-agent
* **Many tools** (8-10+): The LLM loses track of which tool to use
* **Multiple domains**: Billing, shipping, technical need different expertise
* **Long instructions**: System prompt exceeds ~2000 tokens
* **Team ownership**: Different teams own different domains

> This demo focuses on building a strong single agent. Demo 6 covers multi-agent orchestration.

## Building a Single Agent

### Concepts
A production single agent has three key pieces:

1. **System Prompt**: Defines the agent's persona, rules, and available tools
   - "You are a customer support agent. You can look up tickets, check priorities, and count tickets by category."

2. **Tool Definitions**: UC functions the agent can call
   - Each tool has a name, description, and parameter schema
   - The LLM uses the description to decide which tool to call

3. **Agent Loop**: The reason-act-observe cycle
   - The agent may call multiple tools in sequence before responding
   - Each tool result feeds back into the reasoning for the next step

> On Databricks, UC functions provide governed, auditable tools. The agent discovers them through their UC descriptions.

In [0]:
# Setup: Install agents SDK, configure Databricks, create @function_tool tools
# The OpenAI Agents SDK works on Databricks via Chat Completions API.

%pip install openai-agents --quiet 2>&1 | tail -1

import os
from openai import AsyncOpenAI
from agents import Agent, Runner, function_tool
from agents.models.openai_chatcompletions import OpenAIChatCompletionsModel
from databricks.sdk.core import Config

# Configure Databricks as the LLM provider
cfg = Config()
headers = cfg.authenticate()
token = headers['Authorization'].replace('Bearer ', '')
os.environ["OPENAI_API_KEY"] = token
os.environ["OPENAI_BASE_URL"] = f"{cfg.host}/serving-endpoints"

client = AsyncOpenAI(api_key=os.environ["OPENAI_API_KEY"], base_url=os.environ["OPENAI_BASE_URL"])
MODEL = "databricks-meta-llama-3-3-70b-instruct"
model = OpenAIChatCompletionsModel(model=MODEL, openai_client=client)

# Create @function_tool tools wrapping our UC functions
@function_tool
def get_ticket(ticket_id: str) -> str:
    """Returns the subject and description of a support ticket"""
    return spark.sql(f"SELECT module5a_demo5.agent_tools.get_ticket('{ticket_id}')").collect()[0][0]

@function_tool
def count_tickets_by_category(category: str) -> int:
    """Returns the number of tickets in a given category (billing, shipping, technical, general)"""
    return spark.sql(f"SELECT module5a_demo5.agent_tools.count_tickets_by_category('{category}')").collect()[0][0]

@function_tool
def get_priority(ticket_id: str) -> str:
    """Returns the priority of a support ticket (high, medium, low)"""
    return spark.sql(f"SELECT module5a_demo5.agent_tools.get_priority('{ticket_id}')").collect()[0][0]

# Create ONE Agent with all 3 tools
# The agent decides which tool to call based on the question - NO hardcoding
support_agent = Agent(
    name="SupportAgent",
    instructions="You are a customer support agent. Use your tools to answer questions about support tickets. Be concise.",
    model=model,
    tools=[get_ticket, count_tickets_by_category, get_priority]
)

print("=== Agent Created ===")
print(f"Agent: {support_agent.name}")
print(f"Model: {MODEL}")
print(f"Tools: {[t.name for t in support_agent.tools]}")
print()
print("The agent will decide which tool to call for each question.")

Note: you may need to restart the kernel using %restart_python or dbutils.library.restartPython() to use updated packages.
=== Agent Created ===
Agent: SupportAgent
Model: databricks-meta-llama-3-3-70b-instruct
Tools: ['get_ticket', 'count_tickets_by_category', 'get_priority']

The agent will decide which tool to call for each question.


In [0]:
# Test Q1: Ask agent about a specific ticket
# The agent decides which tool to call - NO hardcoding!

print("USER: What is ticket TKT-003 about?")
print()

result = await Runner.run(support_agent, "What is ticket TKT-003 about?")
print(f"Agent: {result.final_output}")
print()

# The agent should have used get_ticket tool
if 'access' in result.final_output.lower() or '403' in result.final_output.lower():
    print("✅ VALIDATION: Agent used get_ticket tool correctly")
else:
    print("❌ VALIDATION: Check agent response")

USER: What is ticket TKT-003 about?

Agent: Ticket TKT-003 is about a user being unable to access their account due to a 403 error when trying to log in. The subject of the ticket is "Cannot access my account" and the description provides more details about the issue.

✅ VALIDATION: Agent used get_ticket tool correctly


In [0]:
# Test Q2: Ask agent to count by category
# The agent decides which tool to call - NO hardcoding!

print("USER: How many billing tickets do we have?")
print()

result = await Runner.run(support_agent, "How many billing tickets do we have?")
print(f"Agent: {result.final_output}")
print()

# The agent should have used count_tickets_by_category tool
if '2' in result.final_output:
    print("✅ VALIDATION: Agent used count_tickets_by_category tool correctly")
else:
    print("❌ VALIDATION: Check agent response")

USER: How many billing tickets do we have?



[non-fatal] Tracing client error 401. Response data is redacted.
[non-fatal] Tracing client error 401. Response data is redacted.


Agent: We have 2 billing tickets.

✅ VALIDATION: Agent used count_tickets_by_category tool correctly


In [0]:
# Test Q3: Ask agent about priority
# The agent decides which tool to call - NO hardcoding!

print("USER: What is the priority of TKT-002?")
print()

result = await Runner.run(support_agent, "What is the priority of TKT-002?")
print(f"Agent: {result.final_output}")
print()

# The agent should have used get_priority tool
if 'medium' in result.final_output.lower():
    print("✅ VALIDATION: Agent used get_priority tool correctly")
else:
    print("❌ VALIDATION: Check agent response")

USER: What is the priority of TKT-002?



[non-fatal] Tracing client error 401. Response data is redacted.


Agent: The priority of ticket TKT-002 is medium.

✅ VALIDATION: Agent used get_priority tool correctly


## Learning Conclusion

### What we demonstrated

| Topic | What was demoed | Key Takeaway |
|---|---|---|
| Anatomy | Reason-act-observe loop | An agent is a sequence of decisions, not a single LLM call |
| 5.1 | Single vs. multi-agent decision | Start with single agent; split when tools >8 or domains differ |
| Building | Full single agent with system prompt + UC tools | System prompt + UC functions + reasoning loop = a working agent |
| Validation | 3 test questions, agent picked correct tool each time | Clear tool descriptions enable correct tool selection |

### Key principles
* **An agent is a loop, not a call**: Reason -> Act -> Observe -> Respond, repeating as needed.
* **UC functions are governed tools**: Register in Unity Catalog for production agents.
* **Tool selection quality depends on descriptions**: Clear COMMENT fields = correct tool picks.
* **Single agents handle complex tasks**: One agent with 3 tools answered 3 different question types correctly.

In [0]:
# CLEANUP: Drop all resources created in this demo
print("Dropping UC functions...")
for fn in ['get_ticket', 'count_tickets_by_category', 'get_priority']:
    spark.sql(f"DROP FUNCTION IF EXISTS module5a_demo5.agent_tools.{fn}")
print("  Functions dropped")

print("Dropping schema and catalog...")
spark.sql("DROP SCHEMA IF EXISTS module5a_demo5.agent_tools CASCADE")
spark.sql("DROP CATALOG IF EXISTS module5a_demo5 CASCADE")
print("  Schema and catalog dropped")

print("\nCleanup complete!")